In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np
import json

In [ ]:
fontsize_legend = 16
fontsize_label = 22
fontsize_ticks = 18

In [ ]:
method2name = {
    'Random': 'Random',
    'RoundRobin': 'RoundRobin',
}
method2color = {
    'Random': [.8, .8, 0],
    'RoundRobin': [1, .6, 0],
    'STOP-V=1000': [1, 0, .3],
}

In [ ]:
legend_elements = [
    plt.Rectangle((0,0), 1, 1, facecolor=[.8, .8, 0], label='Random'),
    plt.Rectangle((0,0), 1, 1, facecolor=[1, .6, 0], label='RoundRobin'),
    plt.Rectangle((0,0), 1, 1, facecolor=[1, 0, .3], label='STOP'),
    plt.Rectangle((0,0), 1, 1, facecolor=[0, .6, .6], label='eGLB'),
    plt.Rectangle((0,0), 1, 1, facecolor=[0, .5, 1], label=r'eGLB_U'),
]
fig = plt.figure(figsize=(len(legend_elements), 1))
plt.legend(handles=legend_elements, loc='center', ncol=len(legend_elements), fontsize=22)
plt.axis('off')
# os.makedirs('plots', exist_ok=True)
# fig.savefig('plots/legend.pdf', bbox_inches='tight')
# plt.show()
plt.close(fig)

In [ ]:
def get_path(model, method, n_m, max_d, id, C_avg=None, R_avg=None):
    path = f'results_retry/{model}/{method}/n_m={n_m}-max_d={max_d}'
    if 'STOP' in method or 'Li' in method:
        path += f'/C_avg={C_avg}-R_avg={R_avg}'
    path += f'/id={id}'
    return path

In [ ]:
def get_data(path):
    data = dict()
    for t in range(300):
        if os.path.isfile(f'{path}/t={t}.json'):
            with open(f'{path}/t={t}.json', 'r') as f:
                d = json.load(f)
            keys = list(d.keys())
            for key in keys:
                if key not in data:
                    data[key] = list()
                data[key].append(d[key])
        else:
            print(f'File {path}/t={t}.json does not exist.')
            break
    for key in keys:
        try:
            data[key] = np.asarray(data[key]).astype('float32')
        except:
            pass

    return {'n_demands': data['arrivals'], 'dispatcher_queue': data['not_executed']}

In [ ]:
def open_data(model, methods, machines, seeds, max_d, C_avg, R_avg):
    method2n_m2data = dict()
    for method in methods:
        method2n_m2data[method] = dict()
        for n_m in machines:
            method2n_m2data[method][n_m] = dict()
            for s in seeds:
                d = get_data(get_path(model, method, n_m, max_d, s, C_avg, R_avg))
                for k in d:
                    if k not in method2n_m2data[method][n_m]:
                        method2n_m2data[method][n_m][k] = list()
                    method2n_m2data[method][n_m][k].append(d[k])
    return method2n_m2data

In [ ]:
machines = range(3, 8)
data = open_data('Yolo', ['STOP-V=1000'], machines, range(30), 1000, .003, 7)
for n_m in machines:
    data['STOP-V=1000'][n_m]['n_demands'] = np.asarray(data['STOP-V=1000'][n_m]['n_demands'])
    data['STOP-V=1000'][n_m]['dispatcher_queue'] = np.asarray(data['STOP-V=1000'][n_m]['dispatcher_queue'])
data.keys(), data['STOP-V=1000'].keys(), data['STOP-V=1000'][3].keys(), data['STOP-V=1000'][3]['n_demands'].shape, data['STOP-V=1000'][3]['dispatcher_queue'].shape

In [ ]:
n_m = 3
plt.figure(figsize=(8, 1))
mean = data['STOP-V=1000'][n_m]['dispatcher_queue'].mean(0)
std = data['STOP-V=1000'][n_m]['dispatcher_queue'].std(0)
plt.plot(mean, label='Dispatcher Queue')
plt.fill_between(range(300), mean - std, mean + std, alpha=0.2)
plt.ylim([0, None])
plt.legend()
plt.show()
plt.close()

In [ ]:
# Little's law
N = data['STOP-V=1000'][n_m]['n_demands'][:, :290].mean(1)
arrival_rate = (data['STOP-V=1000'][n_m]['n_demands'][:, :290] - data['STOP-V=1000'][n_m]['dispatcher_queue'][:, :290]).mean(1)
(N / arrival_rate - 1).mean().item()